# Week 56, Day 3: Generative classifiers: LDA, QDA and naive Bayes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w056/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: LDA from means and a pooled covariance

Write lda_fit(X, y), returning the weight vector and intercept (with equal class priors, b = -w · (mu0 + mu1) / 2). Compare LDA and penalised logistic regression on Gaussian classes with correlated features, at 30 and at 2,000 training trials, averaging test accuracy over 20 training sets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(563)
D = 10
L = np.linalg.cholesky(0.6 * np.ones((D, D)) + 0.4 * np.eye(D))     # correlated features
mu_diff = np.r_[0.8, -0.5, 0.3, np.zeros(D - 3)]

def draw(n, g):
    y = g.integers(0, 2, n)
    X = (g.normal(size=(n, D)) @ L.T) + np.outer(y - 0.5, mu_diff)
    return X, y

def lda_fit(X, y):
    m0, m1 = X[y == 0].mean(0), X[y == 1].mean(0)
    S = (np.cov(X[y == 0].T) * (np.sum(y == 0) - 1) + np.cov(X[y == 1].T) * (np.sum(y == 1) - 1)) / (len(y) - 2)
    w = ...
    b = ...
    return w, b

def logreg_fit(X, y, lam=1e-2):
    A = np.c_[np.ones(len(y)), X]
    def f(w):
        p = np.clip(1 / (1 + np.exp(-A @ w)), 1e-12, 1 - 1e-12)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)) + 0.5 * lam * w[1:] @ w[1:], A.T @ (p - y) / len(y) + lam * np.r_[0, w[1:]]
    w = minimize(f, np.zeros(D + 1), jac=True).x
    return w[1:], w[0]

X_te, y_te = draw(5000, r)
X_tr, y_tr = draw(200, r)
acc = {}
for n in [30, 2000]:
    a_l, a_g = [], []
    for rep in range(20):
        Xn, yn = draw(n, r)
        for fitter, store in [(lda_fit, a_l), (logreg_fit, a_g)]:
            w, b = fitter(Xn, yn); store.append(np.mean((X_te @ w + b > 0) == (y_te == 1)))
    acc[n] = (np.mean(a_l), np.mean(a_g))
    print(f"{n:5d} training trials: LDA {acc[n][0]:.3f}, logistic regression {acc[n][1]:.3f}")

In [ ]:
import numpy as np
def test_fit():
    "lda_fit returns w = Sigma^-1 (mu1 - mu0) with the pooled covariance, and the matching intercept"
    w, b = lda_fit(X_tr, y_tr)
    m0, m1 = X_tr[y_tr == 0].mean(0), X_tr[y_tr == 1].mean(0)
    Sp = (np.cov(X_tr[y_tr == 0].T) * (np.sum(y_tr == 0) - 1) + np.cov(X_tr[y_tr == 1].T) * (np.sum(y_tr == 1) - 1)) / (len(y_tr) - 2)
    assert np.allclose(w, np.linalg.solve(Sp, m1 - m0))
def test_small():
    "with 30 training trials LDA is within 2% of logistic regression"
    assert abs(acc[30][0] - acc[30][1]) < 0.02
def test_large():
    "with 2,000 training trials they are within 1% of each other"
    assert abs(acc[2000][0] - acc[2000][1]) < 0.01

test_fit()
test_small()
test_large()
print("All checks passed")

### Exercise: When the classes differ in spread

Write gauss_logpdf(X, mu, S) for each row of X. Fit QDA (a mean and covariance per class) to two-dimensional classes with the same mean in one direction but a three-fold difference in spread, and compare its test accuracy with LDA's.

In [ ]:
def gauss_logpdf(X, mu, S):
    d = X - mu
    sol = np.linalg.solve(S, d.T).T
    return ...

def two_classes(n, g):
    y = g.integers(0, 2, n)
    X = g.normal(size=(n, 2)) * np.where(y[:, None] == 1, 3.0, 1.0) + np.c_[0.5 * y, np.zeros(n)]
    return X, y

Xq, yq = two_classes(400, r); Xq_te, yq_te = two_classes(5000, r)
params = [(Xq[yq == k].mean(0), np.cov(Xq[yq == k].T)) for k in (0, 1)]
score = gauss_logpdf(Xq_te, *params[1]) - gauss_logpdf(Xq_te, *params[0])
acc_qda = np.mean((score > 0) == (yq_te == 1))
w, b = lda_fit(Xq, yq)
acc_lda = np.mean((Xq_te @ w + b > 0) == (yq_te == 1))
print(f"LDA accuracy {acc_lda:.3f}; QDA accuracy {acc_qda:.3f}")

In [ ]:
import numpy as np
def test_logpdf():
    "gauss_logpdf matches scipy's multivariate normal"
    from scipy.stats import multivariate_normal
    S = np.array([[1.0, 0.3], [0.3, 2.0]]); x = np.array([[0.2, -0.4]])
    assert abs(gauss_logpdf(x, np.zeros(2), S)[0] - multivariate_normal(np.zeros(2), S).logpdf(x[0])) < 1e-9
def test_better():
    "QDA beats LDA by at least 5 points when one class is three times as spread"
    assert acc_qda > acc_lda + 0.05

test_logpdf()
test_better()
print("All checks passed")

### Exercise: A Poisson naive Bayes decoder and its confidence

Estimate each neuron's mean count for each class from the training trials. Write nb_posterior(X, rates), the naive Bayes posterior with equal priors. Compare mean confidence (largest posterior) with accuracy for test trials with and without a shared gain fluctuation.

In [ ]:
K8, N8 = 8, 60
dirs8 = np.arange(K8) * 2 * np.pi / K8
pref8 = r.uniform(0, 2 * np.pi, N8)

def trials_counts(n, g, shared=0.0):
    lab = g.integers(0, K8, n)
    gain = np.exp(shared * g.normal(size=(n, 1)) - shared ** 2 / 2)      # a common fluctuation in excitability
    lam = 2.0 * (1 + 0.8 * np.cos(dirs8[lab][:, None] - pref8)) * gain
    return g.poisson(lam).astype(float), lab

def nb_posterior(X, rates):
    ll = X @ np.log(rates).T - rates.sum(1)          # n x K log-likelihoods
    ll -= ll.max(1, keepdims=True)
    P = np.exp(ll)
    return ...

for name, shared in [("independent", 0.0), ("correlated", 0.5)]:
    Xc, lc = trials_counts(800, r, shared); Xc_te, lc_te = trials_counts(3000, r, shared)
    rates = ...
    P = nb_posterior(Xc_te, rates)
    acc_, conf_ = np.mean(P.argmax(1) == lc_te), P.max(1).mean()
    if shared == 0: acc_ind, conf_ind = acc_, conf_
    else: acc_corr, conf_corr = acc_, conf_
    print(f"{name:12s} noise: accuracy {acc_:.3f}, mean confidence {conf_:.3f}")

In [ ]:
import numpy as np
def test_post():
    "nb_posterior returns probabilities over 8 classes that sum to one"
    P = nb_posterior(Xc_te[:5], rates)
    assert P.shape == (5, 8) and np.allclose(P.sum(1), 1)
def test_overconfident():
    "with correlated noise the decoder's mean confidence exceeds its accuracy by at least 0.05"
    assert conf_corr - acc_corr > 0.05
def test_indep():
    "with independent noise its confidence is within 0.05 of its accuracy"
    assert abs(conf_ind - acc_ind) < 0.05

test_post()
test_overconfident()
test_indep()
print("All checks passed")